# Overview of Training Loss per Component

The BioJEPA-AC model has 3 major components for prediction, and 1 major evaluation head. Each of these components is trained separately and provides a unique component of the model's ability to predict cell states and changes based on a given perturbation. Since we have such a unique architecture, we'll walk through each component's loss evaluation and what is driving the model learning. For our comparative loss elements (L1, NLL, MSE) we only evaluate known gene positions and ignore the unknown positions since we don't actually have a ground truth to compare against.

In [1]:
import numpy as np
import torch.nn.functional as F
import torch

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

## Encoder
![encoder loss](../resources/v0_7/loss_encoder.png)

The first major component in our model is the [cell state encoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_cell_state_encoder_v0_7.ipynb). The cell state encoder learns the unified latent space into which our cells get embedded and across which perturbations move those representations. To train this space, we use an exponential-moving-average target encoder (teacher) in our self-supervised training loop. The EMA momentum anneals to 1 as training progresses so that the teacher stops moving and presents a stable target for the student. The target encoder helps the model avoid representation collapse without needing negative pairs, showing that a slow-moving target network alone provides enough signal asymmetry.

During training, our context encoder (student) receives a masked version of the cell state while the target encoder sees the complete unmasked cell state and provides stable target latents for the context encoder to predict. Both have the unknown genes masked. We don't use gradients to update the teacher's weights. Instead, as you'll see, we use the exponential moving average of the student's weights. This gives the context encoder a slowly-evolving, low-noise target that prevents representation collapse and smooths the training signal.

For our loss analysis, we use a combination of L1 (Mean Absolute Error) loss for masked positions, L2 (Mean Squared Error) loss on context positions, and VICReg loss on all positions. L1 loss drives the model to accurately reconstruct the teacher's latents at masked positions. L2 loss evaluates the accuracy of our context positions and penalizes inaccuracies quadratically, something we do because the model should know the context input and have an easier time predicting it. We use VICReg loss to prevent the representation space from collapsing by ensuring each feature dimension maintains variance and stays decorrelated from the others.

We'll start by staging the outputs of the context encoder and target encoder. We'll also have a mask index that shows which gene positions were masked per sample.

### Data Prep

As part of the forward training pass for our context encoder, to ensure our inputs are masked, we calculate a set of mask indices. We also pass in the per-sample unknown indices. These will be used to ensure we only properly split up our masked, context, and unknowns. The forward pass then creates three latents:
1. `context_latents` created by the input of the masked sample data passing through the context encoder
2. `predicted_latents` created by the masked predictor processing the context latents
3. `target_latents` created by the input of the unmasked sample data passing through the target encoder.

Since we have a notebook explaining how this data is created, we'll focus on staging the latents and mask, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss.

In [2]:
batch = 2
num_genes = 9
embed_dim = 6   

**Mask Index**

Our first component is the boolean mask identifying which genes were masked for each sample in the batch. You'll see that the mask pattern is different per sample. During input, masking clears out the expression value for each masked gene, but for our loss calculation, it identifies the full embedding vector for each masked gene position.

In [3]:
mask_idx=torch.tensor([
    [True,False,False,False,True,True,False,False,True],
    [True,False,False,True,False,False,True,False,False]
])
mask_idx

tensor([[ True, False, False, False,  True,  True, False, False,  True],
        [ True, False, False,  True, False, False,  True, False, False]])

**Unknown Mask**

Our next component is the boolean mask identifying which genes are unmeasured (unknown) for each sample in the batch. You'll see that the mask pattern is different per sample. This is the same mask that was used for the forward pass. We'll actually create two versions of this: the unknown mask to highlight unknown genes, and the gene mask to highlight known positions.

In [4]:
unknown_mask = torch.tensor([
    [False, True, False, False, False, False, False, False, True],
    [False, False, True, False, False, False, False, False, False]
])
gene_mask = ~unknown_mask
unknown_mask.shape, unknown_mask, gene_mask

(torch.Size([2, 9]),
 tensor([[False,  True, False, False, False, False, False, False,  True],
         [False, False,  True, False, False, False, False, False, False]]),
 tensor([[ True, False,  True,  True,  True,  True,  True,  True, False],
         [ True,  True, False,  True,  True,  True,  True,  True,  True]]))

**Mask Combinations**

Now we need to use these two masks to figure out the combination of what are true masked positions (masked and not unknown) and what is our context (not masked and not unknown). This will help us calculate the different components of loss.

In [5]:
is_masked = mask_idx & gene_mask
is_masked

tensor([[ True, False, False, False,  True,  True, False, False, False],
        [ True, False, False,  True, False, False,  True, False, False]])

In [6]:
is_context = ~mask_idx & gene_mask
is_context

tensor([[False, False,  True,  True, False, False,  True,  True, False],
        [False,  True, False, False,  True,  True, False,  True,  True]])

**Context Latents**

These are the generated latents based on the masked input. These are output by the context encoder.

In [7]:
context_latents=torch.tensor([
    [[4.0,3.0,5.1,2.0,6.0,1.0],
    [5.0,4.0,5.1,3.0,7.0,2.0],
    [3.0,2.0,5.1,1.0,5.0,3.0],
    [6.0,5.0,5.1,4.0,8.0,1.0],
    [4.0,3.0,5.4,2.0,6.0,2.0],
    [5.0,4.0,5.1,3.0,7.0,1.0],
    [3.0,2.0,5.2,1.0,4.0,3.0],
    [3.8,2.1,5.3,1.1,2.2,3.0],
    [7.0,6.0,5.1,5.0,9.0,2.0]],
    
    [[2.0,1.0,4.1,3.0,5.0,1.0],
    [6.0,5.0,5.1,4.0,8.0,2.0],
    [4.0,3.0,5.1,2.0,6.0,3.0],
    [3.0,2.0,5.1,1.0,4.0,1.0],
    [5.0,4.0,5.1,3.0,7.0,2.0],
    [7.0,6.0,5.1,5.0,9.0,3.0],
    [4.0,3.0,5.1,2.0,6.0,1.0],
    [2.0,0.1,5.0,6.1,2.2,4.0],
    [2.0,1.0,5.3,1.0,3.0,2.0]],
])
context_latents

tensor([[[4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.4000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.2000, 1.0000, 4.0000, 3.0000],
         [3.8000, 2.1000, 5.3000, 1.1000, 2.2000, 3.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 2.0000]],

        [[2.0000, 1.0000, 4.1000, 3.0000, 5.0000, 1.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [2.0000, 0.1000, 5.0000, 6.1000, 2.2000, 4.00

**Predicted Latents**

These are the predicted latents based on the context latents. These are output by the masked predictor.

In [8]:
predicted_latents=torch.tensor([
    [[4.2,2.8,5.3,1.8,5.7,1.1],
    [4.7,4.3,4.6,3.2,7.3,1.9],
    [3.3,1.7,5.4,0.8,4.8,3.2],
    [5.8,5.2,4.8,4.1,8.2,0.9],
    [3.8,3.2,5.2,2.1,6.3,2.1],
    [5.3,3.7,5.3,2.8,6.8,1.2],
    [2.7,2.3,4.7,1.2,4.2,2.8],
    [2.8,2.2,5.0,1.2,2.1,3.1],
    [6.8,6.2,4.9,5.2,9.3,1.8]],

    [[2.3,0.7,5.3,2.8,4.7,1.2],
    [5.8,5.2,4.8,4.2,8.3,1.8],
    [3.7,3.3,4.7,2.2,6.3,2.8],
    [3.3,1.7,5.3,0.8,3.7,1.2],
    [4.8,4.2,4.8,3.2,7.3,1.8],
    [6.7,6.3,4.7,5.2,9.3,2.8],
    [4.3,2.7,5.3,1.8,5.7,1.2],
    [2.1,1.1,4.7,5.2,3.0,4.1],
    [2.3,0.7,5.3,0.8,2.7,2.2]],
])
predicted_latents

tensor([[[4.2000, 2.8000, 5.3000, 1.8000, 5.7000, 1.1000],
         [4.7000, 4.3000, 4.6000, 3.2000, 7.3000, 1.9000],
         [3.3000, 1.7000, 5.4000, 0.8000, 4.8000, 3.2000],
         [5.8000, 5.2000, 4.8000, 4.1000, 8.2000, 0.9000],
         [3.8000, 3.2000, 5.2000, 2.1000, 6.3000, 2.1000],
         [5.3000, 3.7000, 5.3000, 2.8000, 6.8000, 1.2000],
         [2.7000, 2.3000, 4.7000, 1.2000, 4.2000, 2.8000],
         [2.8000, 2.2000, 5.0000, 1.2000, 2.1000, 3.1000],
         [6.8000, 6.2000, 4.9000, 5.2000, 9.3000, 1.8000]],

        [[2.3000, 0.7000, 5.3000, 2.8000, 4.7000, 1.2000],
         [5.8000, 5.2000, 4.8000, 4.2000, 8.3000, 1.8000],
         [3.7000, 3.3000, 4.7000, 2.2000, 6.3000, 2.8000],
         [3.3000, 1.7000, 5.3000, 0.8000, 3.7000, 1.2000],
         [4.8000, 4.2000, 4.8000, 3.2000, 7.3000, 1.8000],
         [6.7000, 6.3000, 4.7000, 5.2000, 9.3000, 2.8000],
         [4.3000, 2.7000, 5.3000, 1.8000, 5.7000, 1.2000],
         [2.1000, 1.1000, 4.7000, 5.2000, 3.0000, 4.10

**Target Latents**

These are the target latents based on the full input. These are output by the target encoder.

In [9]:
target_latents=torch.tensor([
    [[4.0,3.0,5.0,2.0,6.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [3.0,2.0,5.0,1.0,5.0,3.0],
    [6.0,5.0,5.0,4.0,8.0,1.0],
    [4.0,3.0,5.0,2.0,6.0,2.0],
    [5.0,4.0,5.0,3.0,7.0,1.0],
    [3.0,2.0,5.0,1.0,4.0,3.0],
    [3.1,2.2,5.1,1.2,2.1,3.5],
    [7.0,6.0,5.0,5.0,9.0,2.0]],
    
    [[2.5,1.0,5.0,3.0,5.0,1.2],
    [6.0,5.0,5.0,4.0,8.0,2.0],
    [4.0,3.0,5.0,2.0,6.0,3.0],
    [3.0,2.0,5.0,1.0,4.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [7.0,6.0,5.0,5.0,9.0,3.0],
    [4.0,3.0,5.0,2.0,6.0,1.0],
    [1.7,0.2,5.0,4.2,2.3,3.7],
    [2.0,1.0,5.0,1.0,3.0,2.0]],
])
target_latents

tensor([[[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
         [3.1000, 2.2000, 5.1000, 1.2000, 2.1000, 3.5000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 2.0000]],

        [[2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [1.7000, 0.2000, 5.0000, 4.2000, 2.3000, 3.70

### L1 Loss
Our first component in our total encoder training loss is L1 loss. L1 loss is the mean absolute difference between predicted and target values, penalizing errors proportionally to their magnitude without squaring them. We calculate it as:
$$
\mathcal{L}_{L1} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_{i}|
$$

We only compute L1 on known masked positions because those are the genes the context encoder never saw, so predicting them tests whether the model has learned meaningful relationships between genes rather than just copying its input. Because of this, we actually compare the masked positions from our predicted latents against our target.

We'll start by indexing out the masked positions. You'll see this masking removes the batch dimension and just returns the array of embeddings for the masked positions. Also, because we're only using the masked known gene positions, we get 6 positions to compare instead of the 7 masked positions.

In [10]:
pred_masked = predicted_latents[is_masked]
pred_masked.shape, pred_masked

(torch.Size([6, 6]),
 tensor([[4.2000, 2.8000, 5.3000, 1.8000, 5.7000, 1.1000],
         [3.8000, 3.2000, 5.2000, 2.1000, 6.3000, 2.1000],
         [5.3000, 3.7000, 5.3000, 2.8000, 6.8000, 1.2000],
         [2.3000, 0.7000, 5.3000, 2.8000, 4.7000, 1.2000],
         [3.3000, 1.7000, 5.3000, 0.8000, 3.7000, 1.2000],
         [4.3000, 2.7000, 5.3000, 1.8000, 5.7000, 1.2000]]))

In [11]:
target_masked = target_latents[is_masked]
target_masked.shape, target_masked

(torch.Size([6, 6]),
 tensor([[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 1.0000],
         [2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000]]))

**Calculate L1 Loss**

Now we're ready to calculate the loss on the masked positions between the predicted and the target. If you look closely, because of how we staged our data, most of our loss is fairly consistent across all positions resulting in a relatively low loss.

In [12]:
rec_loss = F.l1_loss(pred_masked, target_masked)
rec_loss

tensor(0.2333)

### MSE (L2) Loss
Our second component is the context L2 loss. L2 loss is the mean squared difference between predicted and target values, penalizing larger errors quadratically. We calculate it as:
$$
\mathcal{L}_{L2} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_{i})^2
$$

We compute L2 only on context positions, the genes the encoder actually saw unmasked. This encourages the masked predictor to also faithfully reconstruct the positions it had access to, not just the ones it had to guess. We have to be careful to not introduce this loss too early as the model might optimize only on learning the context. To avoid this, we keep our context coefficient at 0 until the last 20% (or whatever the configuration is) of training steps, at which point we linearly ramp it up to the configured target. This means for the first 80% of training the model only optimizes masked prediction and using the embedding space, then gradually adds pressure to also reconstruct the unmasked positions correctly as we go through our final learning stages.

We'll start by indexing out the context positions. Like before, you'll see this masking removes the batch dimension and just returns the array of embeddings for the context positions. You'll see that we end up with 9 genes here, meaning that we have 15 covered between the masking (6) and the context (9), leaving the remaining 3 as our unknowns, matching the data setup.

In [13]:
pred_context = predicted_latents[is_context]
pred_context.shape, pred_context

(torch.Size([9, 6]),
 tensor([[3.3000, 1.7000, 5.4000, 0.8000, 4.8000, 3.2000],
         [5.8000, 5.2000, 4.8000, 4.1000, 8.2000, 0.9000],
         [2.7000, 2.3000, 4.7000, 1.2000, 4.2000, 2.8000],
         [2.8000, 2.2000, 5.0000, 1.2000, 2.1000, 3.1000],
         [5.8000, 5.2000, 4.8000, 4.2000, 8.3000, 1.8000],
         [4.8000, 4.2000, 4.8000, 3.2000, 7.3000, 1.8000],
         [6.7000, 6.3000, 4.7000, 5.2000, 9.3000, 2.8000],
         [2.1000, 1.1000, 4.7000, 5.2000, 3.0000, 4.1000],
         [2.3000, 0.7000, 5.3000, 0.8000, 2.7000, 2.2000]]))

In [14]:
targ_context = target_latents[is_context]
targ_context.shape, targ_context

(torch.Size([9, 6]),
 tensor([[3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 1.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
         [3.1000, 2.2000, 5.1000, 1.2000, 2.1000, 3.5000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 2.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
         [1.7000, 0.2000, 5.0000, 4.2000, 2.3000, 3.7000],
         [2.0000, 1.0000, 5.0000, 1.0000, 3.0000, 2.0000]]))

**Calculate L2 Loss**

Now we're ready to calculate the loss on the context positions between the predicted and the target. If you look closely, because of how we staged our data, our loss will be very low. We would expect in general context loss to be low since the positions are known on input.

In [15]:
context_loss = F.mse_loss(pred_context, targ_context)
context_loss

tensor(0.1011)

### Variance-Invariance-Covariance Regularization (VICReg) Loss
VICReg is a regularization technique that prevents representational collapse, where the encoder learns to map all inputs to similar outputs to trivially minimize reconstruction loss. The variance component ensures each feature dimension maintains healthy variance by penalizing any dimension that flattens below std of 1, while the covariance component penalizes correlations between feature dimensions. We compute each component for both the context (x) and target (y) latents and sum them:
$$
\mathcal{L}_{VICReg} = \lambda_{std} \cdot \underbrace{\left(\frac{1}{d} \sum_{j=1}^{d} \text{ReLU}(1 - \sigma_{x,j}) + \frac{1}{d} \sum_{j=1}^{d} \text{ReLU}(1 - \sigma_{y,j})\right)}_{\text{std loss}} + \lambda_{cov} \cdot \underbrace{\left(\frac{1}{d} \sum_{i \neq j} C_{x,ij}^{2} + \frac{1}{d} \sum_{i \neq j} C_{y,ij}^{2}\right)}_{\text{cov loss}}
$$
We weight the standard deviation loss at 25:1 relative to the covariance loss ($\lambda_{std}=25$, $\lambda_{cov}=1$). This means the model gets a much stronger signal to maintain variance than to decorrelate, which is intentional since collapse (all dimensions going flat) is a more catastrophic failure mode than redundancy (two dimensions being correlated).

Since VICReg is about ensuring we use the full embedding space, we compare the context and target latents across all positions, including unknown positions.

In [16]:
std_coeff = 25.0
cov_coeff = 1.0

**Flatten Batch and Genes**

We start by merging the batch and gene dimensions so each gene position across both samples becomes a row. This reshapes our $[B, \text{num\_genes}, \text{embed\_dim}]$ tensors to $[B \times \text{num\_genes}, \text{embed\_dim}]$, giving us 18 rows to compute statistics over.

In [17]:
cont_x = context_latents.reshape(-1, embed_dim).float()
targ_y = target_latents.reshape(-1, embed_dim).float()
B = cont_x.shape[0]
B, cont_x.shape, cont_x, targ_y

(18,
 torch.Size([18, 6]),
 tensor([[4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.4000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.2000, 1.0000, 4.0000, 3.0000],
         [3.8000, 2.1000, 5.3000, 1.1000, 2.2000, 3.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 2.0000],
         [2.0000, 1.0000, 4.1000, 3.0000, 5.0000, 1.0000],
         [6.0000, 5.0000, 5.1000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.1000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.1000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.1000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.1000, 2.0000, 6.0000, 1.0000],
         [2.0000, 0.1000, 5.0

#### Standard Deviations Loss

Our first component to calculate is the standard deviation loss. For this we'll calculate the standard deviation of each embedding dimension across all gene positions, then sum the penalties from both the context and target latents. Notice that we're looking at the embedding channel since the goal is to balance how well it is distributing data across the channels. If it gets too low, we have an issue.

We'll start by calculating the standard deviation for each latent. Notice that in particular our third channel is low. In fact, our target actually has no variation and is only pulled away from 0 variance because of our epsilon.

In [18]:
std_cont = torch.sqrt(cont_x.var(dim=0) + 0.0001)
std_cont

tensor([1.5948, 1.6969, 0.2642, 1.5707, 2.1166, 0.9376])

In [19]:
std_targ = torch.sqrt(targ_y.var(dim=0) + 0.0001)
std_targ

tensor([1.6042, 1.6832, 0.0256, 1.3776, 2.1169, 0.9290])

**Stdev Loss**

Now we're ready to compare our standard deviations to calculate a single loss that penalizes low-variance dimensions. Since the goal is to penalize deviations below 1, we first push anything above 1 negative via $1 - \text{stdev}$ and take the ReLU calculated as $\text{ReLU}(x) = \max(0, x)$. ReLU makes this a one-sided penalty since features with std below 1 get pushed up, but features with std above 1 are left alone. This prevents us from penalizing high-variance features. We then sum the context and target losses together.

In [20]:
stdloss_cont = (1 - std_cont)
stdloss_targ = (1 - std_targ)
stdloss_cont, stdloss_targ

(tensor([-0.5948, -0.6969,  0.7358, -0.5707, -1.1166,  0.0624]),
 tensor([-0.6042, -0.6832,  0.9744, -0.3776, -1.1169,  0.0710]))

In [21]:
stdloss_cont = F.relu(stdloss_cont)
stdloss_targ = F.relu(stdloss_targ)
stdloss_cont, stdloss_targ

(tensor([0.0000, 0.0000, 0.7358, 0.0000, 0.0000, 0.0624]),
 tensor([0.0000, 0.0000, 0.9744, 0.0000, 0.0000, 0.0710]))

In [22]:
stdloss_cont = torch.mean(stdloss_cont)
stdloss_targ = torch.mean(stdloss_targ)
stdloss_cont, stdloss_targ

(tensor(0.1330), tensor(0.1742))

In [23]:
std_loss = stdloss_cont + stdloss_targ
std_loss

tensor(0.3073)

#### Covariance Loss

Our second component penalizes correlations between embedding dimensions. We'll compute the covariance matrix across genes for each latent, then sum up the squared off-diagonal entries. The diagonal represents each dimension's variance with itself, which we already handle with std loss, so we only care about the off-diagonals. If two dimensions are correlated, the model is encoding redundant information and wasting capacity.

You'll notice that features 0 and 1 move in lockstep (feature 1 is always feature 0 minus 1), so their off-diagonal entry will be large. Features 3 and 4 also track each other. The covariance loss will push the model to decorrelate these pairs and spread information more independently across the embedding space.

We'll first mean-center our latents.

In [24]:
cont_x = cont_x - cont_x.mean(dim=0)
cont_x

tensor([[-0.2111, -0.1222,  0.0167, -0.7333,  0.2000, -1.0556],
        [ 0.7889,  0.8778,  0.0167,  0.2667,  1.2000, -0.0556],
        [-1.2111, -1.1222,  0.0167, -1.7333, -0.8000,  0.9444],
        [ 1.7889,  1.8778,  0.0167,  1.2667,  2.2000, -1.0556],
        [-0.2111, -0.1222,  0.3167, -0.7333,  0.2000, -0.0556],
        [ 0.7889,  0.8778,  0.0167,  0.2667,  1.2000, -1.0556],
        [-1.2111, -1.1222,  0.1167, -1.7333, -1.8000,  0.9444],
        [-0.4111, -1.0222,  0.2167, -1.6333, -3.6000,  0.9444],
        [ 2.7889,  2.8778,  0.0167,  2.2667,  3.2000, -0.0556],
        [-2.2111, -2.1222, -0.9833,  0.2667, -0.8000, -1.0556],
        [ 1.7889,  1.8778,  0.0167,  1.2667,  2.2000, -0.0556],
        [-0.2111, -0.1222,  0.0167, -0.7333,  0.2000,  0.9444],
        [-1.2111, -1.1222,  0.0167, -1.7333, -1.8000, -1.0556],
        [ 0.7889,  0.8778,  0.0167,  0.2667,  1.2000, -0.0556],
        [ 2.7889,  2.8778,  0.0167,  2.2667,  3.2000,  0.9444],
        [-0.2111, -0.1222,  0.0167, -0.7

In [25]:
targ_y = targ_y - targ_y.mean(dim=0)
targ_y

tensor([[-0.1833, -0.1333, -0.0056, -0.6333,  0.2000, -1.0778],
        [ 0.8167,  0.8667, -0.0056,  0.3667,  1.2000, -0.0778],
        [-1.1833, -1.1333, -0.0056, -1.6333, -0.8000,  0.9222],
        [ 1.8167,  1.8667, -0.0056,  1.3667,  2.2000, -1.0778],
        [-0.1833, -0.1333, -0.0056, -0.6333,  0.2000, -0.0778],
        [ 0.8167,  0.8667, -0.0056,  0.3667,  1.2000, -1.0778],
        [-1.1833, -1.1333, -0.0056, -1.6333, -1.8000,  0.9222],
        [-1.0833, -0.9333,  0.0944, -1.4333, -3.7000,  1.4222],
        [ 2.8167,  2.8667, -0.0056,  2.3667,  3.2000, -0.0778],
        [-1.6833, -2.1333, -0.0056,  0.3667, -0.8000, -0.8778],
        [ 1.8167,  1.8667, -0.0056,  1.3667,  2.2000, -0.0778],
        [-0.1833, -0.1333, -0.0056, -0.6333,  0.2000,  0.9222],
        [-1.1833, -1.1333, -0.0056, -1.6333, -1.8000, -1.0778],
        [ 0.8167,  0.8667, -0.0056,  0.3667,  1.2000, -0.0778],
        [ 2.8167,  2.8667, -0.0056,  2.3667,  3.2000,  0.9222],
        [-0.1833, -0.1333, -0.0056, -0.6

**Latent Covariance**

We can now compute the covariance matrix by taking the dot product of each feature dimension with every other feature dimension across samples. We normalize by $B - 1$ to apply Bessel's correction for unbiased estimation.

In [26]:
cov_cont = (cont_x.T @ cont_x) / (B - 1)
cov_cont.shape, cov_cont

(torch.Size([6, 6]),
 tensor([[ 2.5434,  2.6774,  0.1014,  1.2467,  3.0071, -0.2007],
         [ 2.6774,  2.8795,  0.0969,  1.1357,  3.3459, -0.3425],
         [ 0.1014,  0.0969,  0.0697, -0.0982, -0.0141,  0.0657],
         [ 1.2467,  1.1357, -0.0982,  2.4671,  1.5576,  0.2686],
         [ 3.0071,  3.3459, -0.0141,  1.5576,  4.4800, -0.6588],
         [-0.2007, -0.3425,  0.0657,  0.2686, -0.6588,  0.8791]]))

In [27]:
cov_targ = (targ_y.T @ targ_y) / (B - 1)
cov_targ.shape, cov_targ

(torch.Size([6, 6]),
 tensor([[ 2.5732e+00,  2.6894e+00, -6.3725e-03,  1.5335e+00,  3.1871e+00,
          -3.1275e-01],
         [ 2.6894e+00,  2.8329e+00, -5.4902e-03,  1.4682e+00,  3.2918e+00,
          -3.2627e-01],
         [-6.3725e-03, -5.4902e-03,  5.5555e-04, -8.4314e-03, -2.1765e-02,
           8.3660e-03],
         [ 1.5335e+00,  1.4682e+00, -8.4314e-03,  1.8976e+00,  1.9565e+00,
          -8.6274e-03],
         [ 3.1871e+00,  3.2918e+00, -2.1765e-02,  1.9565e+00,  4.4812e+00,
          -7.0941e-01],
         [-3.1275e-01, -3.2627e-01,  8.3660e-03, -8.6274e-03, -7.0941e-01,
           8.6301e-01]]))

**Off-diagonals**

We're now ready to pull all the values from the off-diagonals into a long list. This will then allow us to calculate the covariance loss across embedding dimensions.

In [28]:
n, m = cov_cont.shape
offd_cont = cov_cont.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_cont

tensor([ 2.6774,  0.1014,  1.2467,  3.0071, -0.2007,  2.6774,  0.0969,  1.1357,
         3.3459, -0.3425,  0.1014,  0.0969, -0.0982, -0.0141,  0.0657,  1.2467,
         1.1357, -0.0982,  1.5576,  0.2686,  3.0071,  3.3459, -0.0141,  1.5576,
        -0.6588, -0.2007, -0.3425,  0.0657,  0.2686, -0.6588])

In [29]:
n, m = cov_targ.shape
offd_targ = cov_targ.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_targ

tensor([ 2.6894, -0.0064,  1.5335,  3.1871, -0.3127,  2.6894, -0.0055,  1.4682,
         3.2918, -0.3263, -0.0064, -0.0055, -0.0084, -0.0218,  0.0084,  1.5335,
         1.4682, -0.0084,  1.9565, -0.0086,  3.1871,  3.2918, -0.0218,  1.9565,
        -0.7094, -0.3127, -0.3263,  0.0084, -0.0086, -0.7094])

**Covariance Loss**

Now that we have our off-diagonal covariance, we're ready to calculate the loss. We first square the off-diagonal covariance values so both positive and negative correlations are penalized equally if they're the same magnitude, and so we quadratically punish bigger correlations. To get the penalty over all pairs, we then sum and normalize by dividing by dimension.

In [30]:
covl_cont = offd_cont.pow_(2).sum().div(embed_dim)
covl_cont

tensor(11.1245)

In [31]:
covl_targ = offd_targ.pow_(2).sum().div(embed_dim)
covl_targ

tensor(12.4232)

In [32]:
cov_loss = covl_cont + covl_targ
cov_loss

tensor(23.5477)

#### VICReg Loss

Now we're ready to calculate the actual VICReg loss. We scale each loss component (standard deviation loss and covariance loss) and then sum them together so we can tune how aggressively each type of collapse is penalized.

In [33]:
scale_std_loss = std_coeff * std_loss 
scale_std_loss

tensor(7.6813)

In [34]:
scale_cov_loss = cov_coeff * cov_loss
scale_cov_loss

tensor(23.5477)

In [35]:
reg_loss = scale_std_loss + scale_cov_loss
reg_loss

tensor(31.2289)

### Total Encoder Loss

Now that we have our masked position L1 loss, our context L2 loss, and our VICReg loss, we're ready to combine them. We'll simulate a late-stage loss where the context coefficient has ramped up fully. The total loss is calculated as:
$$
\mathcal{L}_{encoder} = \lambda_{sim} \cdot \mathcal{L}_{L1} + \lambda_{context} \cdot \mathcal{L}_{L2} + \mathcal{L}_{VICReg}
$$

We use a similarity coefficient to scale the L1 reconstruction loss and the context coefficient to scale the L2 context loss before adding them to VICReg. This balances the reconstruction objective against the context and regularization, ensuring the model learns accurate latent predictions while also maintaining the ability to predict known positions and maintain a healthy embedding space.

In [36]:
sim_coeff = 25.0
context_coeff = 2.0

In [37]:
rec_loss = sim_coeff * rec_loss
rec_loss 

tensor(5.8333)

In [38]:
context_loss = context_coeff * context_loss
context_loss

tensor(0.2022)

In [39]:
encoder_loss = rec_loss + context_loss + reg_loss
encoder_loss

tensor(37.2645)

### Target Encoder EMA Update

After we update the weights of the context encoder, we also need to update the target encoder or else the model will just converge and learn nothing. The update is based on the exponential moving average (EMA) where we blend a portion of the student's current weights into the teacher with each step, using $\theta_{\text{teacher}} \leftarrow m \cdot \theta_{\text{teacher}} + (1 - m) \cdot \theta_{\text{student}}$ where $m$ is the EMA momentum. We use a default momentum of $0.995$ so the target encoder is moving significantly slower than the student. If you think through this, with a learning rate scheduler, as we get towards the later stages and the learning rate is annealing (aka dropping), the student is moving away from the teacher more slowly so the student and teacher should be coming closer together. In our model, we also anneal the momentum during the later stages, linearly interpolating from the initial momentum $0.995$ toward a final target $1.000$ so that the teacher stops moving and the student has a stable target for the final polishing steps.

## Action Composer
![composer loss](../resources/v0_7/loss_composer.png)

To handle different perturbations, we have the [action composer](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_action_composer_v0_7.ipynb). The composer projects different modalities of perturbations, based on their sequence and/or target, into a unified space so that we can pass them into the AC predictor to shift the cell state.

To train the action composer we use contrastive learning to ensure that the sequence path and target path representations for the same perturbation are close while representations of different perturbations are far apart. For each perturbation, we do a separate sequence and target encoding per sample. We L2-normalize both vectors, then calculate InfoNCE loss which is the cross-entropy over the cosine similarity matrix scaled by a fixed temperature. Note that we're comparing a single perturbation, so in datasets where samples have multiple perturbations, for the Action Composer training, we actually treat each one separately as we're trying to just learn a perturbation representation, and not its cell impact.

InfoNCE loss teaches the model that the sequence encoding and target encoding of the same perturbation should be similar, while being dissimilar to every other perturbation in the batch. This forces the composer to learn that a given DNA sequence (or chemical structure) and its protein target are two views of the same underlying perturbation, so at inference time either pathway alone can produce a meaningful action latent.

We'll start by staging our path encoder outputs for the sequence and target.

### Data Prep 

As part of the forward training pass for our action composer, we only train on perturbations where we have both the sequence and target. While inference can support only having one or the other, to create our shared space we need both to complete contrastive learning. As part of the training forward pass, we create:
1. `z_seq` created by passing the sequence through the sequence path encoder
2. `z_target` created by passing the target through the target path encoder

Since we have a notebook explaining how this data is created, we'll focus on staging the latents, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss. If you look at our staged data, we've purposefully seeded some different values for our second and fourth perturbations to show how the loss is impacted when pairs don't match well.

In [40]:
batch = 4
pert_embed_dims = 8
temperature = 0.012

In [41]:
z_seq=torch.tensor([
    [3.0,1.0,4.0,0.5,2.0,1.0,3.5,0.5],
    [0.5,3.0,1.0,4.0,0.5,2.5,1.0,3.0],
    [2.0,0.5,1.0,3.0,4.0,0.5,2.0,1.0],
    [1.0,2.0,3.0,1.0,0.5,4.0,0.5,2.0],
])
z_seq.shape, z_seq

(torch.Size([4, 8]),
 tensor([[3.0000, 1.0000, 4.0000, 0.5000, 2.0000, 1.0000, 3.5000, 0.5000],
         [0.5000, 3.0000, 1.0000, 4.0000, 0.5000, 2.5000, 1.0000, 3.0000],
         [2.0000, 0.5000, 1.0000, 3.0000, 4.0000, 0.5000, 2.0000, 1.0000],
         [1.0000, 2.0000, 3.0000, 1.0000, 0.5000, 4.0000, 0.5000, 2.0000]]))

In [42]:
z_target=torch.tensor([
    [2.8,1.2,3.7,0.6,2.2,0.8,3.3,0.7],
    [6.7,2.8,1.2,3.8,0.6,0.3,1.2,2.8],
    [1.8,0.7,1.2,2.8,3.7,0.6,2.2,0.8],
    [2.2,1.5,1.1,3.1,3.9,1.5,2.2,1.1],
])
z_target.shape, z_target

(torch.Size([4, 8]),
 tensor([[2.8000, 1.2000, 3.7000, 0.6000, 2.2000, 0.8000, 3.3000, 0.7000],
         [6.7000, 2.8000, 1.2000, 3.8000, 0.6000, 0.3000, 1.2000, 2.8000],
         [1.8000, 0.7000, 1.2000, 2.8000, 3.7000, 0.6000, 2.2000, 0.8000],
         [2.2000, 1.5000, 1.1000, 3.1000, 3.9000, 1.5000, 2.2000, 1.1000]]))

### L2-Normalization

Our first step once we have the path encoder outputs is to normalize them. L2 normalization projects each vector onto the unit hypersphere so that the length of each vector is 1. This ensures that dot products between them become cosine similarities, removing magnitude differences and comparing only directional alignment. Without it, a perturbation with larger embedding norms would dominate the loss logits regardless of semantic similarity. We calculate the normalization as:
$$
\hat{z} = \frac{z}{\|z\|_2} = \frac{z}{\sqrt{\sum_{i=1}^{d} z_i^{2}}}
$$

In [43]:
z_seq = F.normalize(z_seq, dim=1)
z_seq.shape, z_seq

(torch.Size([4, 8]),
 tensor([[0.4536, 0.1512, 0.6047, 0.0756, 0.3024, 0.1512, 0.5292, 0.0756],
         [0.0765, 0.4588, 0.1529, 0.6118, 0.0765, 0.3824, 0.1529, 0.4588],
         [0.3357, 0.0839, 0.1678, 0.5035, 0.6713, 0.0839, 0.3357, 0.1678],
         [0.1678, 0.3357, 0.5035, 0.1678, 0.0839, 0.6713, 0.0839, 0.3357]]))

In [44]:
z_target = F.normalize(z_target, dim=1)
z_target.shape, z_target

(torch.Size([4, 8]),
 tensor([[0.4417, 0.1893, 0.5836, 0.0946, 0.3470, 0.1262, 0.5205, 0.1104],
         [0.7570, 0.3163, 0.1356, 0.4293, 0.0678, 0.0339, 0.1356, 0.3163],
         [0.3155, 0.1227, 0.2104, 0.4909, 0.6486, 0.1052, 0.3857, 0.1402],
         [0.3418, 0.2331, 0.1709, 0.4817, 0.6060, 0.2331, 0.3418, 0.1709]]))

### Temperature-Scaled Cosine Similarity Logits
Now that we've normalized, we're ready to calculate the cosine similarity between the sequence and target. We calculate it as:
$$
\text{logits}_{ij} = \frac{\hat{z}_{seq,i} \cdot \hat{z}_{target,j}}{\tau}
$$
Where $\tau$ is the temperature. We include the division by temperature to sharpen the distribution by scaling up the cosine similarities before they go into cross-entropy, making the model more confident about which pairs match.

The output of this is a $[\text{Batch}, \text{Batch}]$ matrix of the similarity between each sequence and target. Since these are cosine similarities, the values will range from -1 to 1, where higher means more similar. This gives us the benefit of treating them as logits predicting the class. If you look at our second and fourth perturbations, you'll see that the index that corresponds to their position isn't the highest value in the row.

In [45]:
cos_sim = torch.matmul(z_seq, z_target.T) 
cos_sim.shape, cos_sim

(torch.Size([4, 4]),
 tensor([[0.9968, 0.6269, 0.7527, 0.7423],
         [0.4729, 0.6705, 0.6261, 0.7201],
         [0.7466, 0.6665, 0.9959, 0.9753],
         [0.6420, 0.5196, 0.4869, 0.5959]]))

In [46]:
logits = cos_sim / temperature
logits.shape, logits

(torch.Size([4, 4]),
 tensor([[83.0705, 52.2401, 62.7248, 61.8603],
         [39.4047, 55.8715, 52.1710, 60.0051],
         [62.2149, 55.5441, 82.9955, 81.2776],
         [53.5004, 43.2976, 40.5783, 49.6576]]))

### Similarity Labels
Since we're treating the cosine similarities as logits, we need a label to indicate which position in the sequence matches which target in our data. This is as simple as labeling the diagonal.

In [47]:
labels = torch.arange(logits.shape[0])
labels

tensor([0, 1, 2, 3])

### Cross Entropy
Now that we have our similarities as logits and labels, we can calculate the cross entropy. Cross-entropy measures how confidently the model assigns the highest probability (largest logit) to the correct matching pair (label) in each row of the similarity matrix, penalizing it when probability leaks to wrong pairs. It's calculated as:
$$
\mathcal{L}_{align} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp(\text{logits}_{ii})}{\sum_{j=1}^{B} \exp(\text{logits}_{ij})}
$$

Because of the issues we seeded into the second and fourth perturbations, the loss is quite high.

In [48]:
action_condition_loss = F.cross_entropy(logits, labels)
action_condition_loss

tensor(2.0447)

## Action Conditioned (AC) Predictor
![AC predictor loss](../resources/v0_7/loss_acpredictor.png)

How a cell state representation changes in the latent space is controlled by the [Action-Conditioned (AC) predictor](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_ac_predictor_v0_7.ipynb). While the encoder learns a shared unified space for cell representations, the AC predictor learns how to traverse that space given a starting cell state and a perturbation. To train the predictor, we use locked context and target encoders. During this training, we allow the action composer to be fine-tuned and train the dosage layers at 1/10th the predictor's learning rate based on prediction accuracy.

During training, the AC predictor receives the masked and unknown gene flagged context latents (encoded control cells) and the action latent, and predicts the perturbed cell state latent. We also generate a target latent using the target encoder and the case cell. We use a mask annealing schedule to reduce masking over the last few epochs to zero.

For our loss analysis, we use a combination of beta-weighted Gaussian NLL loss and VICReg loss. We run Gaussian NLL on all known gene positions, both masked and context, while we run VICReg on all positions including unknown genes as it's a stabilization force. Gaussian NLL loss drives the predictor to accurately predict the target encoder's latents of the perturbed cell, while also learning calibrated uncertainty estimates through its variance output. We use beta annealing to gradually introduce the down-weighting of high-uncertainty predictions so that early on in training the model focuses on learning the mean. VICReg loss serves the same role as in encoder training, preventing the predicted latent space from collapsing.

We'll start by staging the outputs of the AC predictor and target encoding. We'll also have a mask index that shows which gene positions were masked per sample.

### Data Prep 

As part of the forward training pass for our AC predictor, we calculate a unique set of mask indices to mask the control cell input and pass in the per-sample unknown masks. The forward pass then creates three latents:
1. `pred_mu` representing the average latent of the perturbed cell generated by the AC predictor
2. `pred_logvar` representing the average uncertainty of the perturbed cell generated by the AC predictor
3. `target_latents` created by the input of the unmasked perturbed cell data passing through the target encoder.

Since we have a notebook explaining how this data is created, we'll focus on staging the latents and mask, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss.

**Mask Index**

Our first component is the boolean mask identifying which genes were masked for each sample in the batch. You'll see that the mask pattern is different per sample. During input, masking clears out the expression value for each masked gene, but for our loss calculation, it identifies the full embedding vector for each masked gene position.

In [49]:
mask_idx=torch.tensor([
    [True,False,False,False,True,True,False,False,True],
    [True,False,False,True,False,False,True,False,False]
])
mask_idx

tensor([[ True, False, False, False,  True,  True, False, False,  True],
        [ True, False, False,  True, False, False,  True, False, False]])

**Unknown Mask**

Our next component is the boolean mask identifying which genes are unmeasured (unknown) for each sample in the batch. You'll see that the mask pattern is different per sample. This is the same mask that was used for the forward pass. We'll actually create two versions of this: the unknown mask to highlight unknown genes, and the gene mask to highlight known positions.

In [50]:
unknown_mask = torch.tensor([
    [False, True, False, False, False, False, False, False, True],
    [False, False, True, False, False, False, False, False, False]
])
gene_mask = ~unknown_mask
unknown_mask.shape, unknown_mask, gene_mask

(torch.Size([2, 9]),
 tensor([[False,  True, False, False, False, False, False, False,  True],
         [False, False,  True, False, False, False, False, False, False]]),
 tensor([[ True, False,  True,  True,  True,  True,  True,  True, False],
         [ True,  True, False,  True,  True,  True,  True,  True,  True]]))

**Predicted Mu**

This is the predicted cell latent based on the AC predictor shifting the cell state based on the perturbation.

In [51]:
pred_mu=torch.tensor([
    [[4.1,3.1,5.1,2.1,5.9,1.1],
    [4.9,4.1,4.9,3.1,7.1,2.1],
    [1.1,2.1,5.2,1.1,4.9,2.9],
    [6.1,5.1,5.1,4.1,8.1,1.1],
    [4.1,2.9,5.1,2.1,6.1,2.1],
    [5.1,4.1,5.1,3.1,7.1,1.1],
    [2.9,2.1,5.1,0.9,4.1,3.1],
    [3.6,2.2,5.3,1.1,2.1,3.8],
    [7.1,6.1,5.1,5.1,9.1,2.1]],

    [[4.0,2.5,3.5,1.5,3.5,2.7],
    [6.1,5.1,5.1,4.1,8.1,2.1],
    [5.5,1.5,3.5,3.5,4.5,1.5],
    [4.5,0.5,3.5,2.5,2.5,2.5],
    [5.1,4.1,5.1,3.1,7.1,2.1],
    [5.5,4.5,3.5,3.5,7.5,1.5],
    [4.1,3.1,5.1,2.1,6.1,1.1],
    [2.1,0.1,4.9,6.1,2.3,4.0],
    [3.5,2.5,3.5,2.5,1.5,3.5]],
])
pred_mu.shape, pred_mu

(torch.Size([2, 9, 6]),
 tensor([[[4.1000, 3.1000, 5.1000, 2.1000, 5.9000, 1.1000],
          [4.9000, 4.1000, 4.9000, 3.1000, 7.1000, 2.1000],
          [1.1000, 2.1000, 5.2000, 1.1000, 4.9000, 2.9000],
          [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 1.1000],
          [4.1000, 2.9000, 5.1000, 2.1000, 6.1000, 2.1000],
          [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 1.1000],
          [2.9000, 2.1000, 5.1000, 0.9000, 4.1000, 3.1000],
          [3.6000, 2.2000, 5.3000, 1.1000, 2.1000, 3.8000],
          [7.1000, 6.1000, 5.1000, 5.1000, 9.1000, 2.1000]],
 
         [[4.0000, 2.5000, 3.5000, 1.5000, 3.5000, 2.7000],
          [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 2.1000],
          [5.5000, 1.5000, 3.5000, 3.5000, 4.5000, 1.5000],
          [4.5000, 0.5000, 3.5000, 2.5000, 2.5000, 2.5000],
          [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 2.1000],
          [5.5000, 4.5000, 3.5000, 3.5000, 7.5000, 1.5000],
          [4.1000, 3.1000, 5.1000, 2.1000, 6.1000, 1.1000],
          [2.

**Predicted Log Variance**

This is the predicted uncertainty in the cell latent based on the AC predictor shifting the cell state based on the perturbation. We'll use mainly negative values to indicate high certainty. A logvar of 0 means ±1 which, given that our data is log1p normalized, would be fairly wide uncertainty.

In [52]:
pred_logvar=torch.tensor([
    [[-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-0.4,-0.5,-0.4,-0.5,-0.4,-0.5],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-0.3,-0.5,-0.3,-0.5,-0.3,-0.5],
    [-0.5,-0.4,-0.5,-0.4,-0.5,-0.4],
    [-0.3,-0.5,-0.3,-0.5,-0.3,-0.5],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-1.1,-0.7,-2.1,-1.9,-0.1,0.1],
    [-0.4,-0.5,-0.4,-0.5,-0.4,-0.5]],

    [[-1.8,-2.0,-1.5,-2.0,-1.8,-2.0],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-2.0,-1.8,-2.0,-1.5,-2.0,-1.8],
    [-1.5,-2.0,-1.8,-2.0,-1.5,-2.0],
    [-0.3,-0.5,-0.3,-0.5,-0.3,-0.5],
    [-2.0,-1.5,-2.0,-1.8,-2.0,-1.5],
    [-0.5,-0.3,-0.5,-0.3,-0.5,-0.3],
    [-1.6,-2.2,0.1,-1.0,-0.9,-1.7],
    [-1.8,-2.0,-1.5,-2.0,-1.8,-2.0]],
])
pred_logvar.shape, pred_logvar

(torch.Size([2, 9, 6]),
 tensor([[[-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-0.4000, -0.5000, -0.4000, -0.5000, -0.4000, -0.5000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
          [-0.5000, -0.4000, -0.5000, -0.4000, -0.5000, -0.4000],
          [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-1.1000, -0.7000, -2.1000, -1.9000, -0.1000,  0.1000],
          [-0.4000, -0.5000, -0.4000, -0.5000, -0.4000, -0.5000]],
 
         [[-1.8000, -2.0000, -1.5000, -2.0000, -1.8000, -2.0000],
          [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
          [-2.0000, -1.8000, -2.0000, -1.5000, -2.0000, -1.8000],
          [-1.5000, -2.0000, -1.8000, -2.0000, -1.5000, -2.0000],
          [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
          [-2.0000, -1.5000, -2.0000, -1.8000, -2

**Target Latents**

These are the target latents based on the full perturbed cell input. These are output by the target encoder.

*Note that we reuse the same staged values from the encoder section for simplicity. In actual training, these would be different since the encoder uses the unmasked control cell while full training uses the unmasked perturbed (case) cell.*

In [53]:
target_latents=torch.tensor([
    [[4.0,3.0,5.0,2.0,6.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [3.0,2.0,5.0,1.0,5.0,3.0],
    [6.0,5.0,5.0,4.0,8.0,1.0],
    [4.0,3.0,5.0,2.0,6.0,2.0],
    [5.0,4.0,5.0,3.0,7.0,1.0],
    [3.0,2.0,5.0,1.0,4.0,3.0],
    [3.1,2.2,5.1,1.2,2.1,3.5],
    [7.0,6.0,5.0,5.0,9.0,2.0]],
    
    [[2.5,1.0,5.0,3.0,5.0,1.2],
    [6.0,5.0,5.0,4.0,8.0,2.0],
    [4.0,3.0,5.0,2.0,6.0,3.0],
    [3.0,2.0,5.0,1.0,4.0,1.0],
    [5.0,4.0,5.0,3.0,7.0,2.0],
    [7.0,6.0,5.0,5.0,9.0,3.0],
    [4.0,3.0,5.0,2.0,6.0,1.0],
    [1.7,0.2,5.0,4.2,2.3,3.7],
    [2.0,1.0,5.0,1.0,3.0,2.0]],
])
target_latents

tensor([[[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
         [3.1000, 2.2000, 5.1000, 1.2000, 2.1000, 3.5000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 2.0000]],

        [[2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 2.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 3.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [1.7000, 0.2000, 5.0000, 4.2000, 2.3000, 3.70

### Beta-Weighted Gaussian NLL Loss

Our first component in our total AC predictor training loss is Gaussian NLL loss. The AC predictor outputs both a mean ($\mu$) and log-variance ($\log\sigma^{2}$) per position, so the loss penalizes not just inaccurate predictions on $\mu$ but also miscalibrated confidence. If the model predicts a small variance but is wrong, the loss is severe. Without the beta weighting, the model could cheat by inflating uncertainty on all positions to reduce the squared error term. The beta weighting prevents this by scaling each element's loss by its detached variance raised to $\beta$, down-weighting predictions where the model claims high uncertainty. We anneal $\beta$ from 0 over the first 40% of training steps so the model first learns accurate mean predictions before being held accountable for calibrated confidence. We calculate the beta-weighted Gaussian NLL as:
$$
\mathcal{L}_{\beta\text{-NLL}} = \frac{1}{n} \sum_{i=1}^{n} \sigma_{i}^{2\beta} \cdot \frac{1}{2} \left( \log\sigma_{i}^{2} + \frac{(y_i - \mu_{i})^{2}}{\sigma_{i}^{2}} \right)
$$
We only compute Gaussian NLL on known positions because those are the genes where we have a known baseline and, given we're at the final state, we want to ensure the model learns both the context and masked positions well. Because of this, we compare the known positions from our predicted mean and log-variance against the target encoder's output of the perturbed cell.

You might be wondering why we treat context and masked positions the same in the AC predictor but used a different loss evaluation for each part in the encoder. This is because in the encoder training, the student and teacher see the same cell, so unmasked positions are a trivial preservation task that we need to punish more heavily (L2 loss) while masked positions require genuine reconstruction (L1). In AC predictor training, the target produced by the teacher is from the perturbed cell while the student gets the control cell so they're different cells. This means every position requires a real prediction regardless of masking. A single Gaussian NLL handles all positions uniformly while letting the model express varying confidence through its learned variance.

We'll start by indexing to only the known positions. You'll see this removes the batch dimension and just returns the array of embeddings for the known positions, 15 of the 18 gene embeddings.

In [54]:
pred_mu_masked = pred_mu[gene_mask]
pred_mu_masked.shape, pred_mu_masked

(torch.Size([15, 6]),
 tensor([[4.1000, 3.1000, 5.1000, 2.1000, 5.9000, 1.1000],
         [1.1000, 2.1000, 5.2000, 1.1000, 4.9000, 2.9000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 1.1000],
         [4.1000, 2.9000, 5.1000, 2.1000, 6.1000, 2.1000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 1.1000],
         [2.9000, 2.1000, 5.1000, 0.9000, 4.1000, 3.1000],
         [3.6000, 2.2000, 5.3000, 1.1000, 2.1000, 3.8000],
         [4.0000, 2.5000, 3.5000, 1.5000, 3.5000, 2.7000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 2.1000],
         [4.5000, 0.5000, 3.5000, 2.5000, 2.5000, 2.5000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 2.1000],
         [5.5000, 4.5000, 3.5000, 3.5000, 7.5000, 1.5000],
         [4.1000, 3.1000, 5.1000, 2.1000, 6.1000, 1.1000],
         [2.1000, 0.1000, 4.9000, 6.1000, 2.3000, 4.0000],
         [3.5000, 2.5000, 3.5000, 2.5000, 1.5000, 3.5000]]))

In [55]:
pred_logvar_masked = pred_logvar[gene_mask]
pred_logvar_masked.shape, pred_logvar_masked

(torch.Size([15, 6]),
 tensor([[-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
         [-0.5000, -0.4000, -0.5000, -0.4000, -0.5000, -0.4000],
         [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-1.1000, -0.7000, -2.1000, -1.9000, -0.1000,  0.1000],
         [-1.8000, -2.0000, -1.5000, -2.0000, -1.8000, -2.0000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-1.5000, -2.0000, -1.8000, -2.0000, -1.5000, -2.0000],
         [-0.3000, -0.5000, -0.3000, -0.5000, -0.3000, -0.5000],
         [-2.0000, -1.5000, -2.0000, -1.8000, -2.0000, -1.5000],
         [-0.5000, -0.3000, -0.5000, -0.3000, -0.5000, -0.3000],
         [-1.6000, -2.2000,  0.1000, -1.0000, -0.9000, -1.7000],
         [-1.8000, -2.0000, -1.5000, -2.0000, -1.8000, -2.0000]]))

In [56]:
target_masked = target_latents[gene_mask]
target_masked.shape, target_masked

(torch.Size([15, 6]),
 tensor([[4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 5.0000, 3.0000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 1.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 2.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 1.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 3.0000],
         [3.1000, 2.2000, 5.1000, 1.2000, 2.1000, 3.5000],
         [2.5000, 1.0000, 5.0000, 3.0000, 5.0000, 1.2000],
         [6.0000, 5.0000, 5.0000, 4.0000, 8.0000, 2.0000],
         [3.0000, 2.0000, 5.0000, 1.0000, 4.0000, 1.0000],
         [5.0000, 4.0000, 5.0000, 3.0000, 7.0000, 2.0000],
         [7.0000, 6.0000, 5.0000, 5.0000, 9.0000, 3.0000],
         [4.0000, 3.0000, 5.0000, 2.0000, 6.0000, 1.0000],
         [1.7000, 0.2000, 5.0000, 4.2000, 2.3000, 3.7000],
         [2.0000, 1.0000, 5.0000, 1.0000, 3.0000, 2.0000]]))

**Calculate predicted variance**

Our model outputs log-variance so we need to convert it into variance. We do this by taking the exponential.

In [57]:
variance = torch.exp(pred_logvar_masked)
variance

tensor([[0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.7408, 0.6065, 0.7408, 0.6065, 0.7408, 0.6065],
        [0.6065, 0.6703, 0.6065, 0.6703, 0.6065, 0.6703],
        [0.7408, 0.6065, 0.7408, 0.6065, 0.7408, 0.6065],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.3329, 0.4966, 0.1225, 0.1496, 0.9048, 1.1052],
        [0.1653, 0.1353, 0.2231, 0.1353, 0.1653, 0.1353],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.2231, 0.1353, 0.1653, 0.1353, 0.2231, 0.1353],
        [0.7408, 0.6065, 0.7408, 0.6065, 0.7408, 0.6065],
        [0.1353, 0.2231, 0.1353, 0.1653, 0.1353, 0.2231],
        [0.6065, 0.7408, 0.6065, 0.7408, 0.6065, 0.7408],
        [0.2019, 0.1108, 1.1052, 0.3679, 0.4066, 0.1827],
        [0.1653, 0.1353, 0.2231, 0.1353, 0.1653, 0.1353]])

**Calculate Negative Log-Likelihood (NLL)**

Now that we have the variance, we can calculate the Gaussian NLL using the predicted cell state $\mu$, the target latent, and the variance. Since we have per-gene/embedding dimension variance, we can calculate the loss based on that. As we calculate, you can see that our seventh and last position have significantly higher loss than the other positions across all dimensions. Also, our second row has an issue in the very first position showing the value of having per-embedding position confidence.

In [58]:
nll = F.gaussian_nll_loss(
    pred_mu_masked.float(), 
    target_masked.float(), 
    variance, 
    reduction='none')
nll

tensor([[-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [ 2.7259, -0.1433, -0.2170, -0.1433, -0.2418, -0.1433],
        [-0.1433, -0.2418, -0.1433, -0.2418, -0.1433, -0.2418],
        [-0.2418, -0.1925, -0.2418, -0.1925, -0.2418, -0.1925],
        [-0.1433, -0.2418, -0.1433, -0.2418, -0.1433, -0.2418],
        [-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [-0.1745, -0.3500, -0.8867, -0.9166, -0.0500,  0.0907],
        [ 5.9059,  7.3127,  4.2919,  7.3127,  5.9059,  7.3127],
        [-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [ 4.2919,  7.3127,  5.9059,  7.3127,  4.2919,  7.3127],
        [-0.1433, -0.2418, -0.1433, -0.2418, -0.1433, -0.2418],
        [ 7.3127,  4.2919,  7.3127,  5.9059,  7.3127,  4.2919],
        [-0.2418, -0.1433, -0.2418, -0.1433, -0.2418, -0.1433],
        [-0.4038, -1.0549,  0.0545,  4.4065, -0.4500, -0.6037],
        [ 5.9059,  7.3127,  4.2919,  7.3127,  5.9059,  7.3127]])

**Reconstruction Loss (Beta-weighted NLL)**

Now that we have our NLL, we're ready to add the beta weighting. We generally use a low beta scaler on the variance. The beta-weighted NLL is called reconstruction loss because the model is reconstructing the teacher's latent representation at known gene positions. As a reminder, we start at zero beta so the variance becomes $1$ and does not have meaning, but we ramp up over the training runs. This ensures that the model first focuses on accurate mean predictions, and then, after a bit, calibrated confidence.

In [59]:
beta_nll = 0.10

In [60]:
rec_loss = (nll * variance.detach().pow(beta_nll)).mean() 
rec_loss

tensor(1.2930)

### Variance-Invariance-Covariance Regularization (VICReg) Loss

Our second component of the AC predictor loss is VICReg loss. This is calculated the same way as the encoder VICReg loss, except with the predicted cell state $\mu$ being compared against the target latent. Since the calculation is the same, we'll quickly go through the calculation.

*Note that we share the same coefficients.*

In [61]:
mu_x = pred_mu.reshape(-1, embed_dim).float()
targ_y = target_latents.reshape(-1, embed_dim).float()
B = mu_x.shape[0]
B, mu_x.shape, mu_x, targ_y

(18,
 torch.Size([18, 6]),
 tensor([[4.1000, 3.1000, 5.1000, 2.1000, 5.9000, 1.1000],
         [4.9000, 4.1000, 4.9000, 3.1000, 7.1000, 2.1000],
         [1.1000, 2.1000, 5.2000, 1.1000, 4.9000, 2.9000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 1.1000],
         [4.1000, 2.9000, 5.1000, 2.1000, 6.1000, 2.1000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 1.1000],
         [2.9000, 2.1000, 5.1000, 0.9000, 4.1000, 3.1000],
         [3.6000, 2.2000, 5.3000, 1.1000, 2.1000, 3.8000],
         [7.1000, 6.1000, 5.1000, 5.1000, 9.1000, 2.1000],
         [4.0000, 2.5000, 3.5000, 1.5000, 3.5000, 2.7000],
         [6.1000, 5.1000, 5.1000, 4.1000, 8.1000, 2.1000],
         [5.5000, 1.5000, 3.5000, 3.5000, 4.5000, 1.5000],
         [4.5000, 0.5000, 3.5000, 2.5000, 2.5000, 2.5000],
         [5.1000, 4.1000, 5.1000, 3.1000, 7.1000, 2.1000],
         [5.5000, 4.5000, 3.5000, 3.5000, 7.5000, 1.5000],
         [4.1000, 3.1000, 5.1000, 2.1000, 6.1000, 1.1000],
         [2.1000, 0.1000, 4.9

#### Standard Deviations Loss

In [62]:
std_mu = torch.sqrt(mu_x.var(dim=0) + 0.0001)
std_mu

tensor([1.4761, 1.6061, 0.7391, 1.4030, 2.3396, 0.9383])

In [63]:
std_targ = torch.sqrt(targ_y.var(dim=0) + 0.0001)
std_targ

tensor([1.6042, 1.6832, 0.0256, 1.3776, 2.1169, 0.9290])

**Stdev Loss**

In [64]:
stdloss_mu = (1 - std_mu)
stdloss_targ = (1 - std_targ)
stdloss_mu, stdloss_targ

(tensor([-0.4761, -0.6061,  0.2609, -0.4030, -1.3396,  0.0617]),
 tensor([-0.6042, -0.6832,  0.9744, -0.3776, -1.1169,  0.0710]))

In [65]:
stdloss_mu = F.relu(stdloss_mu)
stdloss_targ = F.relu(stdloss_targ)
stdloss_mu, stdloss_targ

(tensor([0.0000, 0.0000, 0.2609, 0.0000, 0.0000, 0.0617]),
 tensor([0.0000, 0.0000, 0.9744, 0.0000, 0.0000, 0.0710]))

In [66]:
stdloss_mu = torch.mean(stdloss_mu)
stdloss_targ = torch.mean(stdloss_targ)
stdloss_mu, stdloss_targ

(tensor(0.0538), tensor(0.1742))

In [67]:
std_loss = stdloss_mu + stdloss_targ
std_loss

tensor(0.2280)

#### Covariance Loss

In [68]:
mu_x = mu_x - mu_x.mean(dim=0)
mu_x

tensor([[-0.3111,  0.0056,  0.4500, -0.7667,  0.4778, -1.1444],
        [ 0.4889,  1.0056,  0.2500,  0.2333,  1.6778, -0.1444],
        [-3.3111, -0.9944,  0.5500, -1.7667, -0.5222,  0.6556],
        [ 1.6889,  2.0056,  0.4500,  1.2333,  2.6778, -1.1444],
        [-0.3111, -0.1944,  0.4500, -0.7667,  0.6778, -0.1444],
        [ 0.6889,  1.0056,  0.4500,  0.2333,  1.6778, -1.1444],
        [-1.5111, -0.9944,  0.4500, -1.9667, -1.3222,  0.8556],
        [-0.8111, -0.8944,  0.6500, -1.7667, -3.3222,  1.5556],
        [ 2.6889,  3.0056,  0.4500,  2.2333,  3.6778, -0.1444],
        [-0.4111, -0.5944, -1.1500, -1.3667, -1.9222,  0.4556],
        [ 1.6889,  2.0056,  0.4500,  1.2333,  2.6778, -0.1444],
        [ 1.0889, -1.5944, -1.1500,  0.6333, -0.9222, -0.7444],
        [ 0.0889, -2.5944, -1.1500, -0.3667, -2.9222,  0.2556],
        [ 0.6889,  1.0056,  0.4500,  0.2333,  1.6778, -0.1444],
        [ 1.0889,  1.4056, -1.1500,  0.6333,  2.0778, -0.7444],
        [-0.3111,  0.0056,  0.4500, -0.7

In [69]:
targ_y = targ_y - targ_y.mean(dim=0)
targ_y

tensor([[-0.1833, -0.1333, -0.0056, -0.6333,  0.2000, -1.0778],
        [ 0.8167,  0.8667, -0.0056,  0.3667,  1.2000, -0.0778],
        [-1.1833, -1.1333, -0.0056, -1.6333, -0.8000,  0.9222],
        [ 1.8167,  1.8667, -0.0056,  1.3667,  2.2000, -1.0778],
        [-0.1833, -0.1333, -0.0056, -0.6333,  0.2000, -0.0778],
        [ 0.8167,  0.8667, -0.0056,  0.3667,  1.2000, -1.0778],
        [-1.1833, -1.1333, -0.0056, -1.6333, -1.8000,  0.9222],
        [-1.0833, -0.9333,  0.0944, -1.4333, -3.7000,  1.4222],
        [ 2.8167,  2.8667, -0.0056,  2.3667,  3.2000, -0.0778],
        [-1.6833, -2.1333, -0.0056,  0.3667, -0.8000, -0.8778],
        [ 1.8167,  1.8667, -0.0056,  1.3667,  2.2000, -0.0778],
        [-0.1833, -0.1333, -0.0056, -0.6333,  0.2000,  0.9222],
        [-1.1833, -1.1333, -0.0056, -1.6333, -1.8000, -1.0778],
        [ 0.8167,  0.8667, -0.0056,  0.3667,  1.2000, -0.0778],
        [ 2.8167,  2.8667, -0.0056,  2.3667,  3.2000,  0.9222],
        [-0.1833, -0.1333, -0.0056, -0.6

**Latent Covariance**

In [70]:
cov_mu = (mu_x.T @ mu_x) / (B - 1)
cov_mu.shape, cov_mu

(torch.Size([6, 6]),
 tensor([[ 2.1787,  1.7401, -0.0965,  0.9616,  2.3821, -0.8517],
         [ 1.7401,  2.5794,  0.3809,  0.5869,  3.3137, -0.8315],
         [-0.0965,  0.3809,  0.5462,  0.0065,  0.6912, -0.0418],
         [ 0.9616,  0.5869,  0.0065,  1.9682,  1.1620, -0.1420],
         [ 2.3821,  3.3137,  0.6912,  1.1620,  5.4736, -1.6081],
         [-0.8517, -0.8315, -0.0418, -0.1420, -1.6081,  0.8803]]))

In [71]:
cov_targ = (targ_y.T @ targ_y) / (B - 1)
cov_targ.shape, cov_targ

(torch.Size([6, 6]),
 tensor([[ 2.5732e+00,  2.6894e+00, -6.3725e-03,  1.5335e+00,  3.1871e+00,
          -3.1275e-01],
         [ 2.6894e+00,  2.8329e+00, -5.4902e-03,  1.4682e+00,  3.2918e+00,
          -3.2627e-01],
         [-6.3725e-03, -5.4902e-03,  5.5555e-04, -8.4314e-03, -2.1765e-02,
           8.3660e-03],
         [ 1.5335e+00,  1.4682e+00, -8.4314e-03,  1.8976e+00,  1.9565e+00,
          -8.6274e-03],
         [ 3.1871e+00,  3.2918e+00, -2.1765e-02,  1.9565e+00,  4.4812e+00,
          -7.0941e-01],
         [-3.1275e-01, -3.2627e-01,  8.3660e-03, -8.6274e-03, -7.0941e-01,
           8.6301e-01]]))

**Off-diagonals**

In [72]:
n, m = cov_mu.shape
offd_mu = cov_mu.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_mu

tensor([ 1.7401, -0.0965,  0.9616,  2.3821, -0.8517,  1.7401,  0.3809,  0.5869,
         3.3137, -0.8315, -0.0965,  0.3809,  0.0065,  0.6912, -0.0418,  0.9616,
         0.5869,  0.0065,  1.1620, -0.1420,  2.3821,  3.3137,  0.6912,  1.1620,
        -1.6081, -0.8517, -0.8315, -0.0418, -0.1420, -1.6081])

In [73]:
n, m = cov_targ.shape
offd_targ = cov_targ.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()
offd_targ

tensor([ 2.6894, -0.0064,  1.5335,  3.1871, -0.3127,  2.6894, -0.0055,  1.4682,
         3.2918, -0.3263, -0.0064, -0.0055, -0.0084, -0.0218,  0.0084,  1.5335,
         1.4682, -0.0084,  1.9565, -0.0086,  3.1871,  3.2918, -0.0218,  1.9565,
        -0.7094, -0.3127, -0.3263,  0.0084, -0.0086, -0.7094])

**Covariance Loss**

In [74]:
covl_mu = offd_mu.pow_(2).sum().div(embed_dim)
covl_mu

tensor(8.9862)

In [75]:
covl_targ = offd_targ.pow_(2).sum().div(embed_dim)
covl_targ

tensor(12.4232)

In [76]:
cov_loss = covl_mu + covl_targ
cov_loss

tensor(21.4093)

#### VICReg Loss

In [77]:
scale_std_loss = std_coeff * std_loss 
scale_std_loss

tensor(5.6999)

In [78]:
scale_cov_loss = cov_coeff * cov_loss
scale_cov_loss

tensor(21.4093)

In [79]:
reg_loss = scale_std_loss + scale_cov_loss
reg_loss

tensor(27.1093)

### Total Predictor Loss

Now that we have our reconstruction loss and our VICReg loss, we're ready to combine them. We use a similarity coefficient (`sim_coeff`) to scale the reconstruction loss before adding it to VICReg. This balances the reconstruction objective against the regularization, ensuring the model learns accurate latent predictions while also maintaining a healthy embedding space.

In [80]:
rec_loss = sim_coeff * rec_loss
rec_loss 

tensor(32.3244)

In [81]:
predictor_loss = rec_loss + reg_loss
predictor_loss

tensor(59.4336)

## Linear Expression Decoder
Many of our evals rely on having the sample-level expression predicted. We do this using the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders_v0_7.ipynb). To train the decoder, we compare the predicted change in expression (predicted delta) against the real change in expression (real delta).

For our loss analysis, we calculate the mean squared error (MSE). MSE loss is the average of the squared differences between predicted and target values on the known gene positions, penalizing larger errors disproportionately more than smaller ones.

Even though we only evaluate loss on known gene positions, we do expect that our model can predict all gene positions. You'll see that during inference we output all gene positions assuming that as our model sees different examples with different coverage, it can interpolate the unknown positions. The only reason we exclude them in loss is because we don't know what the ground truth for them is so we cannot faithfully evaluate them.

We'll start by staging the outputs of the decoder as the calculated deltas.

### Data Prep

As part of the forward training pass for our linear expression decoder, we go through the full BioJEPA-AC forward pass, then through the decoder. We calculate expression predictions for the control cell and the perturbed cell, then use those to calculate:
1. `pred_delta` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression from the predicted control expression. We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `real_delta` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

Since we have a notebook explaining how this data is created, we'll focus on staging the deltas, and then, from there, go into the loss calculation. We'll stage our data so that we can show how the different loss calculation components impact the total loss.

In [82]:
batch = 2
num_genes = 9

**Predicted Delta**

The predicted delta is the difference between the predicted expression of the perturbed cell and the control cell.

In [83]:
pred_delta=torch.tensor([
    [0.6,9.0,-0.2,1.1,-0.7,0.2,-1.4,0.8,-0.1],
    [0.2,0.3,3.2,-0.4,-0.3,1.1,-0.5,0.2,0.3],
])
pred_delta.shape, pred_delta

(torch.Size([2, 9]),
 tensor([[ 0.6000,  9.0000, -0.2000,  1.1000, -0.7000,  0.2000, -1.4000,  0.8000,
          -0.1000],
         [ 0.2000,  0.3000,  3.2000, -0.4000, -0.3000,  1.1000, -0.5000,  0.2000,
           0.3000]]))

**Real Delta**

The real delta is the difference between the real expression of the perturbed cell and the control cell.

In [84]:
real_delta=torch.tensor([
    [0.5,0.0,-0.3,1.2,-0.8,0.1,-1.5,0.7,0.0],
    [1.0,-3.5,0.0,0.8,-1.2,0.3,0.6,-0.9,1.5],
])

real_delta.shape, real_delta

(torch.Size([2, 9]),
 tensor([[ 0.5000,  0.0000, -0.3000,  1.2000, -0.8000,  0.1000, -1.5000,  0.7000,
           0.0000],
         [ 1.0000, -3.5000,  0.0000,  0.8000, -1.2000,  0.3000,  0.6000, -0.9000,
           1.5000]]))

**Unknown Mask**

Our next component is the boolean mask identifying which genes we have real expression information for and which we don't for each sample in the batch. You'll see that the mask pattern is different per sample. This is the same mask that was used for the forward pass. We'll actually create two versions of this: the unknown mask to highlight unknown genes, and the gene mask to highlight known positions.

In [85]:
unknown_mask = torch.tensor([
    [False, True, False, False, False, False, False, False, True],
    [False, False, True, False, False, False, False, False, False]
])
gene_mask = ~unknown_mask
unknown_mask.shape, unknown_mask, gene_mask

(torch.Size([2, 9]),
 tensor([[False,  True, False, False, False, False, False, False,  True],
         [False, False,  True, False, False, False, False, False, False]]),
 tensor([[ True, False,  True,  True,  True,  True,  True,  True, False],
         [ True,  True, False,  True,  True,  True,  True,  True,  True]]))

### Mean Squared Error (MSE) Loss

We're now ready to calculate the mean squared error loss. The MSE loss is the average of the squared differences between predicted and target values. We calculate it as:
$$
\mathcal{L}_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_{i})^{2}
$$

We'll first apply our masking that will help reduce what are expected errors at the unknown positions since our real_delta treats them as 0.0. The MSE penalizes larger errors disproportionately more than smaller ones. Because of this, you'll see that the second sample in our staged data has a significantly larger error because of the second gene position.

In [86]:
pred_delta = pred_delta[gene_mask]
pred_delta

tensor([ 0.6000, -0.2000,  1.1000, -0.7000,  0.2000, -1.4000,  0.8000,  0.2000,
         0.3000, -0.4000, -0.3000,  1.1000, -0.5000,  0.2000,  0.3000])

In [87]:
real_delta = real_delta[gene_mask]
real_delta

tensor([ 0.5000, -0.3000,  1.2000, -0.8000,  0.1000, -1.5000,  0.7000,  1.0000,
        -3.5000,  0.8000, -1.2000,  0.3000,  0.6000, -0.9000,  1.5000])

In [88]:
decoder_loss = F.mse_loss(pred_delta, real_delta)
decoder_loss

tensor(1.4600)